In [7]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    print(f"val 1 = {x}")
    print(f"central value = {central_value*100:.4f}% \pm {error*100:.4f}%")
    return central_value, error

def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    print(f"val 1 = {x*100:.4f}% \pm {x_err*100:.4f}%")
    print(f"val 2 = {y*100:.4f}% \pm {y_err*100:.4f}%")
    print(f"central value = {central_value*100:.4f}% \pm {error*100:.4f}%")
    return central_value, error

def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)
    print(f"central value = {final_Acp*100:.4f}% \pm {final_Acp_err*100:.4f}%")
    return final_Acp, final_Acp_err

In [9]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    elif A_original_error == A_error:
        delta_Acp_error = 0
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

In [10]:
#Ks, etapip_gg, fitv3 orig.
central_value_1 =  -0.00495716868227114 
stat_unc_1 =  0.0012428556744885479

central_value_2 =  0.009190873073308792
stat_unc_2 =  0.001354329439157585

orig_value, orig_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.4957% \pm 0.1243%
val 2 = 0.9191% \pm 0.1354%
central value = 0.2117% \pm 0.0919%


In [11]:
#Dp_etapip_gg
central_value_1 =  -0.00495716868827234
stat_unc_1 =  0.0012428173613717212

central_value_2 = 0.009190873086162954
stat_unc_2 =  0.001354331286395288

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = -0.4957% \pm 0.1243%
val 2 = 0.9191% \pm 0.1354%
central value = 0.2117% \pm 0.0919%

Original Acp: 0.21169%, Original Acp error: 0.09191%
Acp: 0.21169%, Acp error: 0.09191%
delta_Acp: 0.00000%, delta_Acp_error: 0.00047%


(3.4264813209006206e-12, 4.749482634955872e-06)

In [12]:
#Dsp_etapip_gg
central_value_1 =  -0.004957168714020965
stat_unc_1 =  0.0012428234025946467

central_value_2 = 0.00919087310512623
stat_unc_2 = 0.0013543314103929209 

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = -0.4957% \pm 0.1243%
val 2 = 0.9191% \pm 0.1354%
central value = 0.2117% \pm 0.0919%

Original Acp: 0.21169%, Original Acp error: 0.09191%
Acp: 0.21169%, Acp error: 0.09191%
delta_Acp: 0.00000%, delta_Acp_error: 0.00043%


(3.3806291099836017e-14, 4.326608471654649e-06)

In [15]:
#Ks, etapip_pipipi, fitv3 orig.
central_value_1 =  -0.005620825562001497
stat_unc_1 =   0.0011461459010934164

central_value_2 =  0.009010943177567787
stat_unc_2 =  0.0012540251077938882

orig_value, orig_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.5621% \pm 0.1146%
val 2 = 0.9011% \pm 0.1254%
central value = 0.1695% \pm 0.0849%


In [16]:
#Dp_etapip_pipipi
central_value_1 =  -0.005417565609916353
stat_unc_1 =  0.001158289231931418

central_value_2 = 0.009010943178855868
stat_unc_2 =  0.0012537203742140316

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = -0.5418% \pm 0.1158%
val 2 = 0.9011% \pm 0.1254%
central value = 0.1797% \pm 0.0853%

Original Acp: 0.16951%, Original Acp error: 0.08494%
Acp: 0.17967%, Acp error: 0.08534%
delta_Acp: 0.01016%, delta_Acp_error: 0.00825%


(0.00010162997668661244, 8.249139948645064e-05)

In [17]:
#Dsp_etapip_pipipi
central_value_1 =  -0.005434956800111745
stat_unc_1 =  0.0011659544040495994

central_value_2 = 0.00901094318017348
stat_unc_2 =  0.001253648218220206

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = -0.5435% \pm 0.1166%
val 2 = 0.9011% \pm 0.1254%
central value = 0.1788% \pm 0.0856%

Original Acp: 0.16951%, Original Acp error: 0.08494%
Acp: 0.17880%, Acp error: 0.08560%
delta_Acp: 0.00929%, delta_Acp_error: 0.01059%


(9.293438224772244e-05, 0.0001058939679569158)

In [18]:
#Ks_K, etapip_gg_K, fitv3 orig.
central_value_1 =  -0.000891053282276677
stat_unc_1 = 0.002591316856349276

central_value_2 = 0.01696678351479708  
stat_unc_2 = 0.002695345787618729

orig_value, orig_error= combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = -0.0891% \pm 0.2591%
val 2 = 1.6967% \pm 0.2695%
central value = 0.8038% \pm 0.1869%


In [19]:
#Dp_etapip_gg_K
central_value_1 =  0.00031847388222749906
stat_unc_1 =  0.0025995356026601215

central_value_2 = 0.01754248889696175
stat_unc_2 =  0.002701655038274435

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = 0.0318% \pm 0.2600%
val 2 = 1.7542% \pm 0.2702%
central value = 0.8930% \pm 0.1875%

Original Acp: 0.80379%, Original Acp error: 0.18695%
Acp: 0.89305%, Acp error: 0.18746%
delta_Acp: 0.08926%, delta_Acp_error: 0.01385%


(0.0008926162733344234, 0.00013848585712739997)

In [20]:
#Dsp_etapip_gg_K
central_value_1 =  -0.00023470150441773363
stat_unc_1 =  0.0025705276956967693

central_value_2 = 0.017644823090288364
stat_unc_2 =  0.0026759197412287096

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = -0.0235% \pm 0.2571%
val 2 = 1.7645% \pm 0.2676%
central value = 0.8705% \pm 0.1855%

Original Acp: 0.80379%, Original Acp error: 0.18695%
Acp: 0.87051%, Acp error: 0.18553%
delta_Acp: 0.06672%, delta_Acp_error: 0.02300%


(0.0006671956766751141, 0.00023002873135341686)

In [21]:
#Ks_K, etapip_pipipi_K, fitv3 orig.
central_value_1 =  0.00013218589725561003
stat_unc_1 = 0.002166365637515913

central_value_2 =  0.01753684985818671
stat_unc_2 = 0.0022719620271573465

orig_value, orig_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)

val 1 = 0.0132% \pm 0.2166%
val 2 = 1.7537% \pm 0.2272%
central value = 0.8835% \pm 0.1570%


In [22]:
#Dp_etapip_pipipi_K
central_value_1 =  0.0022384639210168977
stat_unc_1 =  0.0021769453870451997

central_value_2 = 0.01912125680169452
stat_unc_2 =  0.0022811529954881154

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = 0.2238% \pm 0.2177%
val 2 = 1.9121% \pm 0.2281%
central value = 1.0680% \pm 0.1577%

Original Acp: 0.88345%, Original Acp error: 0.15696%
Acp: 1.06799%, Acp error: 0.15766%
delta_Acp: 0.18453%, delta_Acp_error: 0.01482%


(0.0018453424836345489, 0.0001481542091429825)

In [23]:
#Dsp_etapip_pipipi_K
central_value_1 =  0.0009518810033468661
stat_unc_1 =  0.0021462723156826674

central_value_2 = 0.018189290983034567
stat_unc_2 =  0.0022536579445562523

combined_central_value, combined_error = combine_x_plus_y_divided_by_2(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print("")
delta_Acp_sys_unc(orig_value, orig_error, combined_central_value, combined_error)

val 1 = 0.0952% \pm 0.2146%
val 2 = 1.8189% \pm 0.2254%
central value = 0.9571% \pm 0.1556%

Original Acp: 0.88345%, Original Acp error: 0.15696%
Acp: 0.95706%, Acp error: 0.15561%
delta_Acp: 0.07361%, delta_Acp_error: 0.02058%


(0.000736068115469557, 0.00020584736100281596)